# spaCy Language Processing Pipelines
When you call `nlp(text)`, the text passes through a series of **pipeline components** in order. Each component adds annotations to the `Doc`:  
`text → tokenizer → tok2vec → tagger → parser → attribute_ruler → lemmatizer → ner → Doc`

Understanding the pipeline is essential for performance tuning (disabling unused components) and customization (adding/replacing components).

###  Blank Pipeline — Tokenizer Only
With `spacy.blank("en")`, the pipeline is empty (`nlp.pipe_names == []`). Only the tokenizer runs — tokens are created but have no POS tags, lemmas, or entities. This is the fastest option when you only need raw tokens.

In [7]:
import spacy

nlp = spacy.blank("en")

doc = nlp("Captain america ate 100$ of samosa. Then he said I can do this all day.")

for token in doc:
    print(token)

`nlp.pipe_names` returns an empty list `[]` — confirming no processing components are active in the blank model. POS tags and lemmas will not be available.

In [8]:
nlp.pipe_names

###  Full Pre-trained Pipeline — `en_core_web_sm`
Loading a pre-trained model activates 6 components:  
1. **tok2vec** — shared neural encoder (contextual word embeddings)  
2. **tagger** — Part-of-Speech tagging  
3. **parser** — dependency parsing + sentence boundary detection  
4. **attribute_ruler** — rule-based corrections post-tagging  
5. **lemmatizer** — reduces words to their dictionary form  
6. **ner** — Named Entity Recognition

In [9]:
nlp = spacy.load("en_core_web_sm")
nlp.pipe_names

`nlp.pipeline` returns the full list of `(name, component_object)` tuples — useful for introspection and removing specific components with `nlp.remove_pipe(name)`.

In [10]:
nlp.pipeline

###  POS Tagging + Lemmatization
`token.pos_` gives the coarse-grained Part-of-Speech (NOUN, VERB, ADJ...). `spacy.explain()` returns a human-readable description. `token.lemma_` gives the base dictionary form — notice `ate → eat`, `said → say`.

In [11]:
doc = nlp("Captain america ate 100$ of samosa. Then he said I can do this all day.")

for token in doc:
    print(token, "|", spacy.explain(token.pos_), "|", token.lemma_)

###  Named Entity Recognition (NER)
NER identifies **real-world entities** in text and classifies them: `ORG` (organization), `PERSON`, `MONEY`, `GPE` (geopolitical), `DATE`, etc. Access entities via `doc.ents` — each is a `Span` with `.text` and `.label_`.

In [12]:
doc = nlp("Tesla Inc is going to acquire twitter for $45 billion")

for ent in doc.ents:
    print(ent.text, ent.label_)

###  Multi-language Support — French Pipeline
spaCy supports 70+ languages. Each language has its own pre-trained models. For French, download `fr_core_news_sm`:  
```bash
python -m spacy download fr_core_news_sm
```
The entity labels may differ — French models use `PER` instead of `PERSON`, and `MISC` for miscellaneous entities.

In [14]:
nlp = spacy.load("fr_core_news_sm")

`spacy.explain(ent.label_)` interprets entity labels — very useful since label sets differ across languages. Notice `Twitter` is classified as `MISC` in the French model (it lacks a specific ORG category entry).

In [15]:
doc = nlp("Tesla Inc va racheter Twitter pour $45 milliards de dollars")
for ent in doc.ents:
    print(ent.text, " | ", ent.label_, " | ", spacy.explain(ent.label_))

Lemmatization in French works language-specifically — `va → aller` ("goes" → "to go"), `racheter → racheter`. Each language model is trained on its own corpus, so lemma quality varies.

In [16]:
for token in doc:
    print(token, " | ", token.pos_, " | ", token.lemma_)

###  Adding a Single Component to a Blank Pipeline
Need only NER without the full pipeline overhead? Extract a specific component from a loaded model and add it to a blank pipeline using `source=`. This is more memory-efficient than loading the full model.

We load `en_core_web_sm` as the `source`, then `add_pipe("ner", source=source_nlp)` copies only the NER component. Result: `nlp.pipe_names == ['ner']` — a lean pipeline that only does entity recognition.

In [17]:
source_nlp = spacy.load("en_core_web_sm")

nlp = spacy.blank("en")
nlp.add_pipe("ner", source=source_nlp)
nlp.pipe_names

NER still works correctly with only the extracted component. This is the recommended pattern when you need entity recognition at scale without paying the cost of POS tagging and parsing.

In [18]:
doc = nlp("Tesla Inc is going to acquire twitter for $45 billion")
for ent in doc.ents:
    print(ent.text, ent.label_)